## Summary:

1. 590 features, 1 Timestamp, 1 Label columns.
2. No duplicates, 116 columns with zero variance (constant), severe class imbalance (Pass:Failed = 14:1).
3. 28 columns with missing data > 50%, and 4 columns with missing data > 40%.
4. Extreme values observed on multiple columns, with 23 columns outlier > 10%.
5. Dataset span across 337 days.

#### Implementation for Basic Transformation:
1. Encode the timestamp (str) to datetime/timestamp.
2. Exclude 116 columns with zero variance.
3. Consider to drop column with missing data > 40%, consider forward fill on the missing data since involved time.
4. Consider to implement model robust to outlier, or normality assumed model with strategy to outliers (clip max/min value, treat as missing, remove top/bottom 5% value, remove above 3 sigma data).
5. Training model to include imbalance/weighted training, or consider SMOTE.
6. Consider to use temporal split with stratification.

In [53]:
import pandas as pd
import numpy as np
import missingno as msno
from pathlib import Path

curr_dir = Path.cwd()
proj_root = curr_dir.parent
raw_data = proj_root / "data/raw/uci-secom.csv"

df = pd.read_csv(raw_data)

In [54]:
# High Level Overview

print(f"Shape: {df.shape}")
print("-" * 30)
print("Data Type Counts:")
print(df.dtypes.value_counts())
print("-" * 30)

# Identify completely empty or constant columns immediately
nan_cols = df.columns[df.isna().all()].tolist()
const_cols = [c for c in df.columns if df[c].nunique() <= 1]

Shape: (1567, 592)
------------------------------
Data Type Counts:
float64    590
str          1
int64        1
Name: count, dtype: int64
------------------------------


In [55]:
# Type of Data and its Column

print("Type of Data and its Columns")

for data_type in df.dtypes.unique():
    cols = df.select_dtypes(include=data_type).columns
    print(f"--- {data_type} ({len(cols)} columns) ---")
    
    # If there are too many, just show a sample or the count
    if len(cols) > 20:
        print(f"First 10: {list(cols[:10])} ... [plus {len(cols)-10} more]")
    else:
        print(list(cols))
        
    print("-" * 30)

Type of Data and its Columns
--- str (1 columns) ---
['Time']
------------------------------
--- float64 (590 columns) ---
First 10: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9'] ... [plus 580 more]
------------------------------
--- int64 (1 columns) ---
['Pass/Fail']
------------------------------


In [56]:
# Missing Data, Duplicates, Class Imbalance, Zero Variance

print(f"Any Duplicates: {df.duplicated().any()}")
print(f"Columns that are 100% NaN: {len(nan_cols)}")
print(f"Columns with zero variance (constant): {len(const_cols)}")

print("-" * 30)
print(f"Labels Counts:{df['Pass/Fail'].value_counts()}")
print("-" * 30)


Any Duplicates: False
Columns that are 100% NaN: 0
Columns with zero variance (constant): 116
------------------------------
Labels Counts:Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64
------------------------------


In [57]:
# Percentage of Missing Data

missing_count = df.isna().sum()
missing_percentage = (df.isna().sum() / len(df)) * 100

missing_summary = pd.concat([missing_count, missing_percentage], axis=1, keys=['Count', 'Percent'])
print(missing_summary.sort_values(by='Percent', ascending=False).head(50))

     Count    Percent
293   1429  91.193363
292   1429  91.193363
157   1429  91.193363
158   1429  91.193363
492   1341  85.577537
220   1341  85.577537
85    1341  85.577537
358   1341  85.577537
518   1018  64.964901
382   1018  64.964901
245   1018  64.964901
244   1018  64.964901
383   1018  64.964901
384   1018  64.964901
246   1018  64.964901
517   1018  64.964901
110   1018  64.964901
109   1018  64.964901
516   1018  64.964901
111   1018  64.964901
578    949  60.561583
579    949  60.561583
580    949  60.561583
581    949  60.561583
72     794  50.670070
345    794  50.670070
73     794  50.670070
346    794  50.670070
385    715  45.628590
519    715  45.628590
247    715  45.628590
112    715  45.628590
562    273  17.421825
563    273  17.421825
564    273  17.421825
565    273  17.421825
566    273  17.421825
567    273  17.421825
568    273  17.421825
569    273  17.421825
557    260  16.592214
556    260  16.592214
555    260  16.592214
554    260  16.592214
553    260

In [58]:
# Outliers

def get_outlier_summary(data):
    summary_list = []
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    total_rows = len(data)

    for col in numeric_cols:
        # Calculate IQR
        Q1 = data[col].quantile(0.25)
        Q3 = data[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        # Count outliers
        outlier_count = data[(data[col] < lower_bound) | (data[col] > upper_bound)].shape[0]
        outlier_percentage = (outlier_count / total_rows) * 100
        
        summary_list.append({
            'Column': col,
            'Outlier Count': outlier_count,
            'Outlier Percentage': round(outlier_percentage, 2)
        })

    # Create DataFrame and sort
    summary_df = pd.DataFrame(summary_list)
    return summary_df.sort_values(by='Outlier Percentage', ascending=False)

# Execute
outlier_report = get_outlier_summary(df)

In [59]:
print(outlier_report[:30])

    Column  Outlier Count  Outlier Percentage
31      31            354               22.59
40      40            330               21.06
544    544            239               15.25
129    129            238               15.19
312    312            234               14.93
195    195            233               14.87
177    177            230               14.68
448    448            230               14.68
467    467            218               13.91
562    562            215               13.72
251    251            201               12.83
523    523            199               12.70
59      59            196               12.51
41      41            195               12.44
23      23            194               12.38
116    116            192               12.25
389    389            189               12.06
545    545            185               11.81
335    335            180               11.49
471    471            179               11.42
199    199            173         

In [68]:
# 1. Calculate the number of missing values per row
missing_count_row = df.isnull().sum(axis=1)

# 2. Calculate the percentage of missing values per row
# df.shape[1] gives the total number of columns (features)
missing_percentage_row = (missing_count_row / df.shape[1]) * 100

# 3. Combine into a summary DataFrame for easy viewing
row_summary = pd.DataFrame({
    'missing_count': missing_count_row,
    'missing_percentage': missing_percentage_row
})

# 4. Sort to see the "worst" samples (those with most missing data)
print("Top X samples with the most missing data:")
print(row_summary.sort_values(by='missing_percentage', ascending=False).head(20))

Top X samples with the most missing data:
      missing_count  missing_percentage
1566            152           25.675676
1564            148           25.000000
1561            140           23.648649
511             100           16.891892
1152            100           16.891892
810              99           16.722973
93               96           16.216216
95               96           16.216216
814              96           16.216216
995              92           15.540541
1054             92           15.540541
735              88           14.864865
89               87           14.695946
512              84           14.189189
133              84           14.189189
700              84           14.189189
846              84           14.189189
299              80           13.513514
1206             76           12.837838
752              72           12.162162


In [77]:
# Time Frame
new_cols = {
    'timestamp_dt': pd.to_datetime(df['Time']),
    'date_only': pd.to_datetime(df['Time']).dt.date
}

# Add them all at once
df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)

earliest = df['timestamp_dt'].iloc[:, 0].min() if df['timestamp_dt'].ndim > 1 else df['timestamp_dt'].min()
latest = df['timestamp_dt'].iloc[:, 0].max() if df['timestamp_dt'].ndim > 1 else df['timestamp_dt'].max()

print(f"Earliest: {earliest}")
print(f"Latest:   {latest}")

duration = latest - earliest
print(f"Time Span: {duration}")

Earliest: 2008-01-08 02:02:00
Latest:   2008-12-10 18:47:00
Time Span: 337 days 16:45:00
